# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [163]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a consistent plotting style
sns.set_theme(style="whitegrid")

# Display more columns when inspecting the dataset
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [164]:
#load the data and keep a copy of the original
df_raw = pd.read_csv("AviationData.csv", encoding="latin-1", low_memory=False)
df=df_raw.copy(deep=True)

In [165]:
#Step 2: inspect shape, dtypes, NaNs, summary statistics
print(df.shape)
df.info()


(88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88

In [166]:
print(df.isna().mean().sort_values(ascending=False).head(15).round(3))   # share of NaNs per column


Schedule                  0.858
Air.carrier               0.813
FAR.Description           0.640
Aircraft.Category         0.637
Longitude                 0.613
Latitude                  0.613
Airport.Code              0.436
Airport.Name              0.407
Broad.phase.of.flight     0.306
Publication.Date          0.155
Total.Serious.Injuries    0.141
Total.Minor.Injuries      0.134
Total.Fatal.Injuries      0.128
Engine.Type               0.080
Report.Status             0.072
dtype: float64


In [167]:
print(df.describe().T)                                          # numeric summary
print(df.describe(include="object").T)

                          count      mean        std  min  25%  50%  75%  \
Number.of.Engines       82805.0  1.146585   0.446510  0.0  1.0  1.0  1.0   
Total.Fatal.Injuries    77488.0  0.647855   5.485960  0.0  0.0  0.0  0.0   
Total.Serious.Injuries  76379.0  0.279881   1.544084  0.0  0.0  0.0  0.0   
Total.Minor.Injuries    76956.0  0.357061   2.235625  0.0  0.0  0.0  0.0   
Total.Uninjured         82977.0  5.325440  27.913634  0.0  0.0  1.0  2.0   

                          max  
Number.of.Engines         8.0  
Total.Fatal.Injuries    349.0  
Total.Serious.Injuries  161.0  
Total.Minor.Injuries    380.0  
Total.Uninjured         699.0  
                       count unique             top   freq
Event.Id               88889  87951  20001212X19172      3
Investigation.Type     88889      2        Accident  85015
Accident.Number        88889  88863      CEN22LA149      2
Event.Date             88889  14782      1984-06-30     25
Location               88837  27758   ANCHORAGE, AK    4

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [168]:
# Normalise column names (e.g. 'Aircraft.damage' vs 'Aircraft.Damage', stray spaces)
df.columns = [c.strip().replace(" ", ".") for c in df.columns]
cols_lower = {c.lower(): c for c in df.columns}
def col(name):
    
    return cols_lower[name.lower()]
df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [169]:
# Execute filters
date_col = col("Event.Date")
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df["Year"] = df[date_col].dt.year
df = df[df["Year"] >= 1983]

# Professional builds only
df[col("Amateur.Built")] = df[col("Amateur.Built")].astype(str).str.strip().str.title()
df = df[df[col("Amateur.Built")] == "No"]

# Airplanes only (impute missing category from engine type)
cat = col("Aircraft.Category")
df[cat] = df[cat].astype("string").str.strip().str.title()
airplane_engines = {"Reciprocating", "Turbo Prop", "Turbo Fan", "Turbo Jet", "Jet"}
eng_ok = df[col("Engine.Type")].astype("string").str.strip().str.title().isin(
    {e.title() for e in airplane_engines})
df.loc[df[cat].isna() & eng_ok, cat] = "Airplane"
df = df[df[cat] == "Airplane"].copy()
print("After filtering:", df.shape)

After filtering: (67627, 32)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [170]:
inj_cols = [col("Total.Fatal.Injuries"), col("Total.Serious.Injuries"),
            col("Total.Minor.Injuries"), col("Total.Uninjured")]

# Drop events where none of the four injury counts was recorded
df = df.dropna(subset=inj_cols, how="all").copy()

# Remaining NaNs -> 0 people in that category
df[inj_cols] = df[inj_cols].fillna(0)

# Estimated occupants and injury measures
df["Total.Occupants"] = df[inj_cols].sum(axis=1)
df = df[df["Total.Occupants"] > 0].copy()

df["Severe.Injuries"] = df[col("Total.Fatal.Injuries")] + df[col("Total.Serious.Injuries")]
df["Severe.Injury.Rate"] = df["Severe.Injuries"] / df["Total.Occupants"]
df["Fatal.Rate"] = df[col("Total.Fatal.Injuries")] / df["Total.Occupants"]
df["Any.Severe.Injury"] = (df["Severe.Injuries"] > 0).astype(int)

print(df.shape)
df[["Total.Occupants", "Severe.Injury.Rate", "Fatal.Rate", "Any.Severe.Injury"]].describe()

(66705, 37)


,Total.Occupants,Severe.Injury.Rate,Fatal.Rate,Any.Severe.Injury
count,66705.000000,66705.000000,66705.000000,66705.000000
mean,6.863488,0.263928,0.170676,0.302991
std,30.303195,0.425223,0.366093,0.459555
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000
75%,2.000000,0.500000,0.000000,1.000000
max,699.000000,1.000000,1.000000,1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [171]:
dmg = col("Aircraft.Damage")
print(df[dmg].value_counts(dropna=False))

Aircraft.damage
Substantial    49442
Destroyed      13401
Minor           1970
NaN             1818
Unknown           74
Name: count, dtype: int64


In [172]:
df[dmg] = df[dmg].astype("string").str.strip().str.title()
df[dmg] = df[dmg].replace({"Unknown": pd.NA, "Unk": pd.NA, "": pd.NA})

# Drop rows with no usable damage info, then create the flag
df = df.dropna(subset=[dmg]).copy()
df["Destroyed"] = (df[dmg] == "Destroyed").astype(int)

print(df[dmg].value_counts())
print("Share destroyed:", df["Destroyed"].mean().round(3))

Aircraft.damage
Substantial    49442
Destroyed      13401
Minor           1970
Name: count, dtype: Int64
Share destroyed: 0.207


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [173]:
mk = col("Make")
df = df.dropna(subset=[mk]).copy()
print("Unique makes before cleaning:", df[mk].nunique())

df[mk] = (df[mk].astype(str)
         .str.upper()
         .str.replace(r"[.,]", "", regex=True)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip())

make_map = {
    "BEECHCRAFT": "BEECH", "BEECH AIRCRAFT": "BEECH", "BEECH AIRCRAFT CORP": "BEECH",
    "RAYTHEON": "BEECH", "RAYTHEON AIRCRAFT COMPANY": "BEECH",
    "CESSNA AIRCRAFT": "CESSNA", "CESSNA AIRCRAFT COMPANY": "CESSNA",
    "PIPER AIRCRAFT": "PIPER", "PIPER AIRCRAFT INC": "PIPER", "THE NEW PIPER AIRCRAFT INC": "PIPER",
    "NEW PIPER": "PIPER",
    "THE BOEING COMPANY": "BOEING", "BOEING COMPANY": "BOEING", "BOEING OF CANADA": "BOEING",
    "AIRBUS INDUSTRIE": "AIRBUS", "AIRBUS INDUSTRIE SAS": "AIRBUS",
    "MOONEY AIRCRAFT": "MOONEY", "MOONEY AIRPLANE CO INC": "MOONEY",
    "MCDONNELL DOUGLAS": "MCDONNELL DOUGLAS", "DOUGLAS": "MCDONNELL DOUGLAS",
    "MCDONNELL-DOUGLAS": "MCDONNELL DOUGLAS",
    "DEHAVILLAND": "DE HAVILLAND", "DE HAVILLAND CANADA": "DE HAVILLAND",
    "EMBRAER SA": "EMBRAER",
    "AERO COMMANDER": "COMMANDER", "COMMANDER AIRCRAFT": "COMMANDER",
    "GRUMMAN AMERICAN": "GRUMMAN", "AMERICAN GRUMMAN": "GRUMMAN",
    "MAULE AIRCRAFT": "MAULE",
}
df[mk] = df[mk].replace(make_map)
print("Unique makes after cleaning:", df[mk].nunique())



Unique makes before cleaning: 1671
Unique makes after cleaning: 1293


In [174]:
# Keep makes with at least 50 records
make_counts = df[mk].value_counts()
print(make_counts.head(30))

keep_makes = make_counts[make_counts >= 50].index
df = df[df[mk].isin(keep_makes)].copy()
print("Makes kept:", len(keep_makes), "| rows:", df.shape[0])

Make
CESSNA                           25052
PIPER                            13777
BEECH                             5030
GRUMMAN                           1259
MOONEY                            1258
BOEING                            1181
BELLANCA                           966
AIR TRACTOR                        656
AERONCA                            604
ROBINSON                           600
MAULE                              566
BELL                               542
CHAMPION                           500
MCDONNELL DOUGLAS                  469
DE HAVILLAND                       444
STINSON                            421
COMMANDER                          410
LUSCOMBE                           385
TAYLORCRAFT                        363
NORTH AMERICAN                     362
ROCKWELL                           327
HUGHES                             318
CIRRUS DESIGN CORP                 284
SCHWEIZER                          265
AIR TRACTOR INC                    226
AYRES               

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [175]:
mod = col("Model")
df = df.dropna(subset=[mod]).copy()
df[mod] = (df[mod].astype(str).str.upper()
           .str.replace(r"\s+", " ", regex=True).str.strip())
df = df[df[mod] != ""].copy()

# Build unique plane type identifier (Plane.ID)
df["Plane.ID"] = df[mk] + " " + df[mod]
print("Unique plane types (Plane.ID):", df["Plane.ID"].nunique())

print(df[mod].value_counts().head(20))
print("Unique model labels:", df[mod].nunique())

Unique plane types (Plane.ID): 4716
Model
152          2205
172          1602
172N         1070
PA-28-140     851
172M          747
150           738
172P          656
182           599
180           588
150M          548
PA-18-150     544
PA-28-180     543
PA-18         541
PA-28-161     516
PA-28-181     495
150L          430
A36           423
G-164A        421
PA-38-112     416
140           383
Name: count, dtype: int64
Unique model labels: 4380


In [176]:
# Are model labels unique to each make?
makes_per_model = df.groupby(mod)[mk].nunique().sort_values(ascending=False)
print("Model labels used by more than one make:", (makes_per_model > 1).sum())
makes_per_model[makes_per_model > 1].head(15)

Model labels used by more than one make: 257


Model
G-164A    6
G-164     5
G-164C    5
G-164B    5
500       5
7AC       4
S-2R      4
690B      4
112       4
690A      4
500-S     4
AA-5      4
AA-5B     4
7ECA      4
S2R       4
Name: Make, dtype: int64

In [177]:
# Examples of shared labels
shared = makes_per_model[makes_per_model > 1].index[:5]
df[df[mod].isin(shared)].groupby([mod, mk]).size()

Model   Make                          
500     CESSNA                             25
        COMMANDER                          14
        EMBRAER                             1
        NORTH AMERICAN                      1
        ROCKWELL                            2
G-164   GRUMMAN                            85
        GRUMMAN ACFT ENG COR-SCHWEIZER      3
        GRUMMAN AMERICAN AVN CORP           1
        GRUMMAN-SCHWEIZER                   2
        SCHWEIZER                           4
G-164A  AIR TRACTOR                         1
        GRUMMAN                           335
        GRUMMAN ACFT ENG COR-SCHWEIZER     10
        GRUMMAN-SCHWEIZER                  49
        GULFSTREAM                          1
        SCHWEIZER                          25
G-164B  GRUMMAN                           178
        GRUMMAN ACFT ENG COR-SCHWEIZER     24
        GRUMMAN-SCHWEIZER                  62
        GULFSTREAM                          4
        SCHWEIZER                        

In [178]:
# Not unique -> build a unique identifier for each plane type
df["Plane.ID"] = df[mk] + " " + df[mod]
print("Unique plane types:", df["Plane.ID"].nunique())
df["Plane.ID"].value_counts().head(20)

Unique plane types: 4716


Plane.ID
CESSNA 152         2205
CESSNA 172         1602
CESSNA 172N        1070
PIPER PA-28-140     851
CESSNA 172M         747
CESSNA 150          738
CESSNA 172P         656
CESSNA 182          599
CESSNA 180          587
CESSNA 150M         548
PIPER PA-18-150     544
PIPER PA-28-180     543
PIPER PA-18         541
PIPER PA-28-161     516
PIPER PA-28-181     495
CESSNA 150L         430
BEECH A36           423
PIPER PA-38-112     416
CESSNA 140          383
CESSNA 170B         372
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [179]:
other_cols = [col("Engine.Type"), col("Weather.Condition"), col("Number.of.Engines"),
              col("Purpose.of.flight"), col("Broad.phase.of.flight")]
for c in other_cols:
    print("=====", c, "=====")
    print(df[c].value_counts(dropna=False).head(15), "\n")

===== Engine.Type =====
Engine.Type
Reciprocating      53613
Turbo Prop          2563
NaN                 2396
Turbo Fan           1218
Turbo Jet            364
Unknown               72
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

===== Weather.Condition =====
Weather.Condition
VMC    53396
IMC     4675
NaN     1544
UNK      486
Unk      135
Name: count, dtype: int64 

===== Number.of.Engines =====
Number.of.Engines
1.0    50304
2.0     7903
NaN     1468
3.0      279
4.0      277
0.0        5
Name: count, dtype: int64 

===== Purpose.of.flight =====
Purpose.of.flight
Personal               35001
Instructional           8520
Unknown                 3743
Aerial Application      3712
Business                3012
NaN                     2412
Positioning             1017
Other Work Use           601
Ferry                    563
Aerial Observation       456
Executive/corporate      327
Public Aircraft          324
Skydiving           

In [180]:
other_cols = [col("Engine.Type"), col("Weather.Condition"), col("Number.of.Engines"),
              col("Purpose.of.flight"), col("Broad.phase.of.flight")]

# Engine type
eng = col("Engine.Type")
df[eng] = df[eng].astype("string").str.strip().str.title()
df[eng] = df[eng].replace({"Unknown": pd.NA, "Unk": pd.NA, "None": pd.NA, "": pd.NA, "Jet": "Turbo Jet"})


In [181]:
# Weather
wx = col("Weather.Condition")
df[wx] = df[wx].astype("string").str.strip().str.upper()
df[wx] = df[wx].replace({"UNK": pd.NA, "UNKNOWN": pd.NA, "": pd.NA})

In [182]:
# Number of engines
ne = col("Number.of.Engines")
df[ne] = pd.to_numeric(df[ne], errors="coerce")
df.loc[(df[ne] <= 0) | (df[ne] > 4), ne] = np.nan

In [183]:
# Purpose and phase of flight
for c in [col("Purpose.of.flight"), col("Broad.phase.of.flight")]:
    df[c] = df[c].astype("string").str.strip().str.title()
    df[c] = df[c].replace({"Unknown": pd.NA, "Unk": pd.NA, "": pd.NA})

In [184]:
# === STATISTICAL IMPUTATION BLOCK (PUT THIS HERE) ===
print("Missing before strict fill:", df[other_cols].isna().sum())

df['Number.of.Engines'] = df['Number.of.Engines'].fillna(df['Number.of.Engines'].mode()[0])
df['Engine.Type'] = df['Engine.Type'].fillna('Reciprocating')
df['Weather.Condition'] = df['Weather.Condition'].fillna('VMC')
df['Purpose.of.flight'] = df['Purpose.of.flight'].fillna('Personal')
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].fillna('Landing')

print("\nMissing after strict fill:", df[other_cols].isna().sum())

Missing before strict fill: Engine.Type               2469
Weather.Condition         2165
Number.of.Engines         1473
Purpose.of.flight         6155
Broad.phase.of.flight    15187
dtype: int64

Missing after strict fill: Engine.Type              0
Weather.Condition        0
Number.of.Engines        0
Purpose.of.flight        0
Broad.phase.of.flight    0
dtype: int64


In [185]:
# === TEXT & GEOGRAPHIC COLUMNS IMPUTATION BLOCK ===
text_cols = ['Location', 'Country', 'Airport.Code', 'Airport.Name', 'Registration.Number']
print("Missing values before filling text columns:\n", df[text_cols].isna().sum())

# Find the mode for Country, or fallback to 'United States'
top_country = df['Country'].mode()[0] if not df['Country'].mode().empty else 'United States'

# Fill missing values using mode for country and descriptive tags for off-airport/unknowns
df['Country'] = df['Country'].fillna(top_country)
df['Location'] = df['Location'].fillna('Unknown Location')
df['Airport.Code'] = df['Airport.Code'].fillna('OFF-AIRPORT')
df['Airport.Name'] = df['Airport.Name'].fillna('Off-Airport / Private Strip')
df['Registration.Number'] = df['Registration.Number'].fillna('UNKNOWN')

print("\nMissing values after filling text columns:\n", df[text_cols].isna().sum())

Missing values before filling text columns:
 Location                  15
Country                   98
Airport.Code           23551
Airport.Name           21824
Registration.Number      158
dtype: int64

Missing values after filling text columns:
 Location               0
Country                0
Airport.Code           0
Airport.Name           0
Registration.Number    0
dtype: int64


In [186]:
#for c in other_cols:
    #print("=====", c, "=====")
    #print(df[c].value_counts(dropna=False).head(12), "\n")

In [187]:
print("Missing values before statistical imputation:")
print(df[other_cols].isna().sum())

# 1. Number.of.Engines: Impute using group mode per Plane.ID, fallback to global mode
if ne in df.columns:
    df[ne] = df[ne].fillna(
        df.groupby("Plane.ID")[ne].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
    )
    global_engine_mode = df[ne].mode()[0] if not df[ne].mode().empty else 1
    df[ne] = df[ne].fillna(global_engine_mode)

# 2. Categorical Columns: Impute using the statistical Mode (most frequent value)
categorical_stats_fills = {
    col("Engine.Type"): df[col("Engine.Type")].mode()[0] if not df[col("Engine.Type")].mode().empty else "Reciprocating",
    col("Weather.Condition"): df[col("Weather.Condition")].mode()[0] if not df[col("Weather.Condition")].mode().empty else "VMC",
    col("Purpose.of.flight"): df[col("Purpose.of.flight")].mode()[0] if not df[col("Purpose.of.flight")].mode().empty else "Personal",
    col("Broad.phase.of.flight"): df[col("Broad.phase.of.flight")].mode()[0] if not df[col("Broad.phase.of.flight")].mode().empty else "Landing"
}

for c, mode_val in categorical_stats_fills.items():
    if c in df.columns:
        df[c] = df[c].fillna(mode_val)

print("\nMissing values after statistical imputation:")
print(df[other_cols].isna().sum())

Missing values before statistical imputation:
Engine.Type              0
Weather.Condition        0
Number.of.Engines        0
Purpose.of.flight        0
Broad.phase.of.flight    0
dtype: int64

Missing values after statistical imputation:
Engine.Type              0
Weather.Condition        0
Number.of.Engines        0
Purpose.of.flight        0
Broad.phase.of.flight    0
dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [188]:
nan_share = df.isna().mean().sort_values(ascending=False)

In [189]:
# Drop columns with more than 50% NaNs, but protect the columns the analysis needs
protected = ["Year", "Plane.ID", "Total.Occupants", "Severe.Injuries", "Severe.Injury.Rate",
             "Fatal.Rate", "Any.Severe.Injury", "Destroyed"] + other_cols + [mk, mod, dmg]
to_drop = [c for c in nan_share[nan_share > 0.5].index if c not in protected]
print("Dropping columns with> 50%:", to_drop)

df = df.drop(columns=to_drop)
print(df.shape)
df.head()

Dropping columns with> 50%: ['Schedule', 'Air.carrier', 'FAR.Description', 'Longitude', 'Latitude']
(60236, 34)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,Aircraft.Category,Registration.Number,Make,Model,Amateur.Built,Number.of.Engines,Engine.Type,Purpose.of.flight,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Year,Total.Occupants,Severe.Injuries,Severe.Injury.Rate,Fatal.Rate,Any.Severe.Injury,Destroyed,Plane.ID
3601,20001214X42095,Accident,SEA83LA036,1983-01-01,"NEWPORT, OR",United States,ONP,NEWPORT MUNICIPAL,Non-Fatal,Substantial,Airplane,N1296M,CESSNA,182P,No,1.0,Reciprocating,Personal,0.0,0.0,1.0,3.0,VMC,Approach,Probable Cause,NaN,1983,4.0,0.0,0.0,0.0,0,0,CESSNA 182P
3602,20001214X42067,Accident,MKC83LA056,1983-01-01,"WOODBINE, IA",United States,3YR,MUNICIPAL,Non-Fatal,Substantial,Airplane,N2639C,CESSNA,182RG,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,1983,2.0,0.0,0.0,0.0,0,0,CESSNA 182RG
3603,20001214X42063,Accident,MKC83LA050,1983-01-01,"MARYVILLE, MO",United States,78Y,RANKIN,Non-Fatal,Substantial,Airplane,N58664,CESSNA,182P,No,1.0,Reciprocating,Personal,0.0,0.0,0.0,1.0,VMC,Takeoff,Probable Cause,NaN,1983,1.0,0.0,0.0,0.0,0,0,CESSNA 182P
3604,20001214X42018,Accident,LAX83FUG11,1983-01-01,"UPLAND, CA",United States,CCB,CABLE,Non-Fatal,Substantial,Airplane,N3084R,PIPER,PA-28R-200,No,1.0,Reciprocating,Personal,0.0,0.0,2.0,0.0,VMC,Approach,Probable Cause,NaN,1983,2.0,0.0,0.0,0.0,0,0,PIPER PA-28R-200
3605,20001214X41951,Accident,CHI83LA074,1983-01-01,"SPRINGBROOK, WI",United States,OFF-AIRPORT,SPRINGBROOK,Non-Fatal,Substantial,Airplane,N89399,CESSNA,140,No,1.0,Reciprocating,Instructional,0.0,0.0,0.0,2.0,VMC,Landing,Probable Cause,NaN,1983,2.0,0.0,0.0,0.0,0,0,CESSNA 140


In [190]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60236 entries, 3601 to 88886
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                60236 non-null  object        
 1   Investigation.Type      60236 non-null  object        
 2   Accident.Number         60236 non-null  object        
 3   Event.Date              60236 non-null  datetime64[ns]
 4   Location                60236 non-null  object        
 5   Country                 60236 non-null  object        
 6   Airport.Code            60236 non-null  object        
 7   Airport.Name            60236 non-null  object        
 8   Injury.Severity         60236 non-null  object        
 9   Aircraft.damage         60236 non-null  string        
 10  Aircraft.Category       60236 non-null  string        
 11  Registration.Number     60236 non-null  object        
 12  Make                    60236 non-null  object  

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [191]:
df.to_csv("AviationData_clean.csv", index=False)
print("Saved:", df.shape)

Saved: (60236, 34)
